In [1]:
from pathlib import Path
import xarray as xr
import pandas as pd
from neuralhydrology.evaluation.metrics import (
    mean_peak_timing,
    missed_peaks,
    mean_absolute_percentage_peak_error,
)
import matplotlib.pyplot as plt
from xarray import DataArray
import numpy as np

from scipy import signal
from neuralhydrology.datautils import utils
from neuralhydrology.evaluation.metrics import _validate_inputs, _mask_valid

In [17]:
# ------------- Paths -------------
time_series = Path("../../extending_caravan/extended_dataset/data/time_series") #US
# camels_timeseries = Path("/inputs/basin_dataset_public_v1p2/basin_mean_forcing/daymet") #US
basins_file = Path("../../extending_caravan/extended_dataset/basins_subset_test.txt") #US
us_output = Path("./peak_metrics/united_states") #US
us_output.mkdir(parents=True, exist_ok=True)

time_series_uy = Path("../data/time_series") #UY
basins_file_uy = Path("../basins.txt") #UY 
uy_output = Path("./peak_metrics/uruguay") #UY
uy_output.mkdir(parents=True, exist_ok=True)

In [3]:
def custom_mean_peak_timing(obs, sim, window=None, resolution='1D', datetime_coord=None, distance=100):
    _validate_inputs(obs, sim)
    obs, sim = _mask_valid(obs, sim)

    peaks, _ = signal.find_peaks(obs.values, distance=distance, prominence=np.std(obs.values))

    if datetime_coord is None:
        datetime_coord = utils.infer_datetime_coord(obs)
    if window is None:
        window = max(int(utils.get_frequency_factor('12h', resolution)), 3)

    timing_errors = []
    for idx in peaks:
        if (idx - window < 0) or (idx + window >= len(obs)) or (
            pd.date_range(obs[idx - window][datetime_coord].values,
                          obs[idx + window][datetime_coord].values,
                          freq=resolution).size != 2 * window + 1):
            continue

        if (sim[idx] > sim[idx - 1]) and (sim[idx] > sim[idx + 1]):
            peak_sim = sim[idx]
        else:
            values = sim[idx - window:idx + window + 1]
            peak_sim = values[values.argmax()]

        peak_obs = obs[idx]
        delta = peak_obs.coords[datetime_coord] - peak_sim.coords[datetime_coord]
        timing_errors.append(np.abs(delta.values / pd.to_timedelta(resolution)))

    return np.mean(timing_errors) if len(timing_errors) > 0 else np.nan


def custom_missed_peaks(obs, sim, window=None, resolution='1D', percentile=80, datetime_coord=None, distance=30):
    _validate_inputs(obs, sim)
    obs, sim = _mask_valid(obs, sim)

    min_obs_height = np.percentile(obs.values, percentile)
    min_sim_height = np.percentile(sim.values, percentile)

    peaks_obs_times, _ = signal.find_peaks(obs, distance=distance, height=min_obs_height)
    peaks_sim_times, _ = signal.find_peaks(sim, distance=distance, height=min_sim_height)

    if len(peaks_obs_times) == 0:
        return 0.

    if datetime_coord is None:
        datetime_coord = utils.infer_datetime_coord(obs)
    if window is None:
        window = max(int(utils.get_frequency_factor('12h', resolution)), 1)

    missed_events = 0
    for idx in peaks_obs_times:
        if (idx - window < 0) or (idx + window >= len(obs)) or (
            pd.date_range(obs[idx - window][datetime_coord].values,
                          obs[idx + window][datetime_coord].values,
                          freq=resolution).size != 2 * window + 1):
            continue

        nearby_peak_sim_index = np.where(np.abs(peaks_sim_times - idx) <= window)[0]
        if len(nearby_peak_sim_index) == 0:
            missed_events += 1

    return missed_events / len(peaks_obs_times)

def mean_absolute_percentage_peak_error(obs: DataArray, sim: DataArray) -> float:
    r"""Calculate the mean absolute percentage error (MAPE) for peaks

    .. math:: \text{MAPE}_\text{peak} = \frac{1}{P}\sum_{p=1}^{P} \left |\frac{Q_{s,p} - Q_{o,p}}{Q_{o,p}} \right | \times 100,

    where :math:`Q_{s,p}` are the simulated peaks (here, `sim`), :math:`Q_{o,p}` the observed peaks (here, `obs`) and
    `P` is the number of peaks.

    Uses scipy.find_peaks to find peaks in the observed time series. The observed peaks indices are used to subset
    observed and simulated flows. Finally, the MAPE metric is calculated as the mean absolute percentage error
    of observed peak flows and corresponding simulated flows.

    Parameters
    ----------
    obs : DataArray
        Observed time series.
    sim : DataArray
        Simulated time series.

    Returns
    -------
    float
        Mean absolute percentage error (MAPE) for peaks.
    """
    # verify inputs
    _validate_inputs(obs, sim)

    # get time series with only valid observations
    obs, sim = _mask_valid(obs, sim)

    # return np.nan if there are no valid observed or simulated values
    if obs.size == 0 or sim.size == 0:
        return np.nan

    # heuristic to get indices of peaks and their corresponding height.
    peaks, _ = signal.find_peaks(obs.values, distance=100, prominence=np.std(obs.values))

    # check if any peaks exist, otherwise return np.nan
    if peaks.size == 0:
        return np.nan

    # subset data to only peak values
    obs = obs[peaks].values
    sim = sim[peaks].values

    # calculate the mean absolute percentage peak error
    peak_mape = np.sum(np.abs((sim - obs) / obs)) / peaks.size * 100

    return peak_mape

# UY

In [11]:
with open(basins_file_uy, "r") as f:
    basins_uy = f.read().splitlines()

basins_uy

['CAMELS_UY_2',
 'CAMELS_UY_3',
 'CAMELS_UY_5',
 'CAMELS_UY_6',
 'CAMELS_UY_7',
 'CAMELS_UY_8',
 'CAMELS_UY_9',
 'CAMELS_UY_10',
 'CAMELS_UY_11',
 'CAMELS_UY_15',
 'CAMELS_UY_16']

In [21]:
# Precip products to compare against gauge
PRECIP_COLS = {
    "CARAVAN":   "prcp_mm_day",
    "CHIRPS": "prcp_chirps_mm_day",
    "MSWEP":  "prcp_mswep_mm_day",

}
GAUGE_COL = "prcp_gauge_mm_day"

results = []

for basin_name in basins_uy:
    try:
        ds = xr.open_dataset(time_series_uy / f"{basin_name}.nc")
        vars_to_drop = [v for v in ["basin"] if v in ds]
        if vars_to_drop:
            ds = ds.drop_vars(vars_to_drop)

        if GAUGE_COL not in ds:
            print(f"Skipping {basin_name}: no gauge precip")
            continue

        obs: DataArray = ds[GAUGE_COL]

        for product_name, col in PRECIP_COLS.items():
            if col not in ds:
                continue

            sim: DataArray = ds[col]

            row = {"basin": basin_name, "product": product_name}

            try:
                row["peak_timing"] = custom_mean_peak_timing(
                    obs, sim, resolution="1D", datetime_coord="date",
                    distance=100, window=5
                )
            except Exception as e:
                row["peak_timing"] = np.nan
                print(f"  peak_timing failed ({basin_name}, {product_name}): {e}")

            try:
                row["missed_peaks"] = custom_missed_peaks(
                    obs, sim, resolution="1D", datetime_coord="date",
                    distance=100, window=5
                )
            except Exception as e:
                row["missed_peaks"] = np.nan
                print(f"  missed_peaks failed ({basin_name}, {product_name}): {e}")

            try:
                row["mape_peak"] = mean_absolute_percentage_peak_error(obs, sim)
            except Exception as e:
                row["mape_peak"] = np.nan
                print(f"  mape_peak failed ({basin_name}, {product_name}): {e}")

            results.append(row)

    except Exception as e:
        print(f"Skipping {basin_name}: {e}")

# ── Summary ───────────────────────────────────────────────────────────────────
df_results = pd.DataFrame(results)


In [18]:
df_results.to_csv(uy_output / "peak_metrics.csv", index=False)

# US

In [7]:
with open(basins_file, "r") as f:
    basins = f.read().splitlines()

# basins

In [20]:
PRECIP_COLS_US = {
    # "camels": "camels_precipitation",
    "CARAVAN":   "total_precipitation_sum",
    "CHIRPS":   "chirps_precipitation",
    "MSWEP":    "mswep_precipitation",
}
GAUGE_COL_US = "camels_precipitation"

results_us = []

for basin_name in basins:
    basin_id = basin_name.replace("camels_", "")

    try:
        ds = xr.open_dataset(time_series / f"{basin_name}.nc")
        ds = ds.drop_vars("basin")

        obs: DataArray = ds[GAUGE_COL_US]

        for product_name, col in PRECIP_COLS_US.items():
            if col not in ds:
                print(f"  Skipping {product_name} for {basin_id}: column not found")
                continue

            sim: DataArray = ds[col]
            row = {"basin": basin_id, "product": product_name}

            try:
                row["peak_timing"] = custom_mean_peak_timing(
                    obs, sim, resolution="1D", datetime_coord="date",
                    distance=100, window=5
                )
            except Exception as e:
                row["peak_timing"] = np.nan
                print(f"  peak_timing failed ({basin_id}, {product_name}): {e}")

            try:
                row["missed_peaks"] = custom_missed_peaks(
                    obs, sim, resolution="1D", datetime_coord="date",
                    distance=100, window=5
                )
            except Exception as e:
                row["missed_peaks"] = np.nan
                print(f"  missed_peaks failed ({basin_id}, {product_name}): {e}")

            try:
                row["mape_peak"] = mean_absolute_percentage_peak_error(obs, sim)
            except Exception as e:
                row["mape_peak"] = np.nan
                print(f"  mape_peak failed ({basin_id}, {product_name}): {e}")

            results_us.append(row)

    except StopIteration:
        print(f"Skipping {basin_id}: CAMELS txt file not found")
    except Exception as e:
        print(f"Skipping {basin_id}: {e}")

# ── Summary ───────────────────────────────────────────────────────────────────
df_results_us = pd.DataFrame(results_us)

In [19]:
df_results_us.to_csv(us_output / "peak_metrics.csv", index=False)